Model Structure:
I took a pretrained roBERTa model from hugging face and removed the pretraining head and replaced it with one for only 2 output classes which was trained from scratch while fine-tuning the models weights with a low learning rate (2e-5), also using the roBERTa tokenizer I truncated and padded the reviews to 256 tokens.

Small and Imbalanced dataset:
To adjust for the mass majority of negative reviews I weighted the cross entropy loss with 180/60, 60/60 so that the model would not just select negative since that was right a majority of the time. In the split training and validation datasets I did an 80/20 split so the validation is accurate to the training split. The biggest of all was using a pretrained model, since there was such little training data to go off of I decided to use a pretrained model that I could mold into the model that I needed with finetuning. I also used f1 scores to track and checkpoint the model since I noticed that the epochs were fluctuating since the dataset was so small

Training techniques:
Since I was using a pretrained model I did not want a large learning rate because I wanted to tune the weights from before instead of working from scratch, I used the AdamW optimizer after doing research on which to use because AdamW is better at avoiding overfitting which was an issue with a dataset as small as this one. The batch sizes were kept small since there were limited training rows. The epochs were set at 5 as 10 would be too lengthy and testing with 7 resulted in the model plateuing around 5 epochs.

Evaluation results:

                precision    recall  f1-score   support

    negative       0.86      0.60      0.71       200
    positive       0.69      0.91      0.79       200

    accuracy                           0.75       400
    macro avg      0.78      0.75      0.75       400
    weighted avg   0.78      0.75      0.75       400

Confusion Matrix:
 $\begin{bmatrix}120 & 80\\19 & 181\end{bmatrix}$

USAGE OF AI:
Claude Sonnet was used as a debugging assistant for runtime errors and was used to explain mechanics of PyTorch and HuggingFace that I was not familiar with. All design decisions were made by myself.

In [ ]:
import copy
from pathlib import Path


import torch
from torch import nn
from torch.optim import AdamW

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

from scipy.special import softmax
import numpy as np
import pandas as pd

In [ ]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, ignore_mismatched_sizes=True)

In [3]:
DATA_DIR = Path("model_checkpoint")

train = pd.read_csv("train.csv")
test = pd.read_csv("public_test.csv")

In [4]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def tokenize(text):
    return tokenizer(text, padding="max_length", max_length=256, return_tensors="pt", truncation=True)

train_set, validation_set = train_test_split(train, test_size=0.2, stratify=train['label'], random_state=42)

train_encodings = tokenize(train_set['text'].tolist())
validation_encodings = tokenize(validation_set['text'].tolist())

train_set = train_set.reset_index(drop=True)
validation_set = validation_set.reset_index(drop=True)

train_ds = ReviewDataset(train_encodings, train_set['label'].tolist())
validation_ds = ReviewDataset(validation_encodings, validation_set['label'].tolist())

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True)
validation_loader = torch.utils.data.DataLoader(validation_ds, batch_size=16)

In [5]:

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

class_weights = torch.tensor([180/60, 60/60])
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_val_f1 = -1
best_state = None

for epoch in range(5):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = criterion(outputs.logits, batch['labels'])
        loss.backward()
        optimizer.step()

    model.eval()
    predict, labels = [], []
    with torch.no_grad():
        for batch in validation_loader:
            outputs = model(**batch)
            predict.extend(outputs.logits.argmax(-1).tolist())
            labels.extend(batch['labels'].tolist())

    validation_f1 = f1_score(labels, predict, average='macro')
    print(f"epoch {epoch}: validation_f1 {validation_f1:.3f}")

    if validation_f1 > best_val_f1:
        best_val_f1 = validation_f1
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)

epoch 0: validation_f1 0.429
epoch 1: validation_f1 0.644
epoch 2: validation_f1 0.768
epoch 3: validation_f1 0.842
epoch 4: validation_f1 0.757


<All keys matched successfully>

In [6]:

model.save_pretrained("model_checkpoint/")
tokenizer.save_pretrained("model_checkpoint/")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.85it/s]


('model_checkpoint/tokenizer_config.json', 'model_checkpoint/tokenizer.json')

In [12]:
test_set = pd.read_csv("public_test.csv")
test_encodings = tokenize(test_set["text"].tolist())

test_ds = ReviewDataset(test_encodings, test_set["label"].tolist())
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16)



model.eval()
predictions, labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(**batch)
        predictions.extend(outputs.logits.argmax(-1).tolist())
        labels.extend(batch["labels"].tolist())

print(classification_report(labels, predictions, target_names=["negative", "positive"]))
print(confusion_matrix(labels, predictions))

              precision    recall  f1-score   support

    negative       0.86      0.60      0.71       200
    positive       0.69      0.91      0.79       200

    accuracy                           0.75       400
   macro avg       0.78      0.75      0.75       400
weighted avg       0.78      0.75      0.75       400

[[120  80]
 [ 19 181]]


In [14]:

public_test = pd.read_csv("public_test.csv")


test_inputs = tokenizer(public_test['text'].tolist(), padding="max_length", max_length=256, truncation=True, return_tensors="pt")

reloaded_model.eval()


all_predictions = []
batch_size = 16
with torch.no_grad():
    for i in range(0, len(public_test), batch_size):
        batch = {k: v[i:i+batch_size] for k, v in test_inputs.items()}
        outputs = reloaded_model(**batch)
        predictions = outputs.logits.argmax(-1).tolist()
        all_predictions.extend(predictions)


predictions_df = pd.DataFrame({
    "id": public_test["id"],
    "predicted_label": all_predictions,
})

predictions_df.to_csv("public_test_predictions.csv", index=False)
print(predictions_df.head())
print(f"Saved {len(predictions_df)} predictions")

                id  predicted_label
0  pos_cv696_29740                1
1  pos_cv669_22995                1
2   neg_cv963_7208                0
3   pos_cv182_7281                1
4  pos_cv162_10424                1
Saved 400 predictions
